## Étape 1 — Weather-Enhanced Energy Forecasting

### Partie A — Mise en pratique

#### A.1 Préparation des températures

In [2]:
import pandas as pd
import numpy as np
import joblib

weather_path = "data_processed/open-meteo-48.89N2.29E36m.csv"

# Open-Meteo ajoute quelques lignes de métadonnées.
# On cherche automatiquement la vraie ligne d'en-tête.
with open(weather_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

header_row = next(
    i for i, line in enumerate(lines)
    if "time" in line and "temperature_2m" in line
)

weather_raw = pd.read_csv(
    weather_path,
    skiprows=header_row
)

print(weather_raw.head())
print(weather_raw.columns)
print(weather_raw.shape)

   location_id              time  temperature_2m (°C)
0            0  2012-01-01T00:00                 12.0
1            0  2012-01-01T01:00                 12.1
2            0  2012-01-01T02:00                 12.1
3            0  2012-01-01T03:00                 12.1
4            0  2012-01-01T04:00                 12.0
Index(['location_id', 'time', 'temperature_2m (°C)'], dtype='str')
(613680, 3)


In [3]:
cities = {
    0: "Paris",
    1: "Lille",
    2: "Lyon",
    3: "Marseille",
    4: "Bordeaux"
}

In [4]:
temp_col = [
    c for c in weather_raw.columns
    if c.startswith("temperature_2m")
][0]

weather_raw = weather_raw.rename(
    columns={temp_col: "temperature"}
)

weather_raw["city"] = (
    weather_raw["location_id"]
    .map(cities)
)

In [5]:
weather = weather_raw.pivot(
    index="time",
    columns="city",
    values="temperature"
)

weather.index = pd.to_datetime(
    weather.index
)

print(weather.head())
print(weather.isna().mean())

city                 Bordeaux  Lille  Lyon  Marseille  Paris
time                                                        
2012-01-01 00:00:00      12.7   11.2   8.5       12.0   12.0
2012-01-01 01:00:00      12.7   11.2   8.5       11.8   12.1
2012-01-01 02:00:00      12.6   11.1   8.5       11.7   12.1
2012-01-01 03:00:00      12.5   11.0   8.3       11.5   12.1
2012-01-01 04:00:00      12.5   11.2   7.9       11.4   12.0
city
Bordeaux     0.0
Lille        0.0
Lyon         0.0
Marseille    0.0
Paris        0.0
dtype: float64


In [6]:
weather["temp_france"] = weather[
    [
        "Paris",
        "Lille",
        "Lyon",
        "Marseille",
        "Bordeaux"
    ]
].mean(axis=1)

In [7]:
joblib.dump(
    weather,
    "data_processed/weather_france_hourly.joblib"
)

['data_processed/weather_france_hourly.joblib']

In [8]:
temp = weather["temp_france"].copy()

temp_30 = (
    temp
    .resample("30min")
    .interpolate("time")
)

In [9]:
weather_features = pd.DataFrame(
    index=temp_30.index
)

weather_features["temp_lag_1h"] = temp_30.shift(2)

weather_features["temp_lag_24h"] = temp_30.shift(48)

weather_features["temp_lag_7d"] = temp_30.shift(336)

weather_features["temp_mean_24h"] = (
    temp_30.shift(2)
    .rolling(48)
    .mean()
)

weather_features["temp_mean_7d"] = (
    temp_30.shift(2)
    .rolling(336)
    .mean()
)

T = weather_features["temp_lag_1h"]

weather_features["heating_degree"] = (
    15 - T
).clip(lower=0)

weather_features["cooling_degree"] = (
    T - 22
).clip(lower=0)

In [10]:
print(weather.shape)

print(
    weather[
        [
            "Paris",
            "Lille",
            "Lyon",
            "Marseille",
            "Bordeaux",
            "temp_france"
        ]
    ].head()
)

print(weather.isna().mean())

(122736, 6)
city                 Paris  Lille  Lyon  Marseille  Bordeaux  temp_france
time                                                                     
2012-01-01 00:00:00   12.0   11.2   8.5       12.0      12.7        11.28
2012-01-01 01:00:00   12.1   11.2   8.5       11.8      12.7        11.26
2012-01-01 02:00:00   12.1   11.1   8.5       11.7      12.6        11.20
2012-01-01 03:00:00   12.1   11.0   8.3       11.5      12.5        11.08
2012-01-01 04:00:00   12.0   11.2   7.9       11.4      12.5        11.00
city
Bordeaux       0.0
Lille          0.0
Lyon           0.0
Marseille      0.0
Paris          0.0
temp_france    0.0
dtype: float64


In [11]:
temp = weather["temp_france"].copy()

temp_30 = (
    temp
    .resample("30min")
    .ffill()
)

In [12]:
weather_features["temp_lag_1h"] = temp_30.shift(2)
weather_features["temp_lag_24h"] = temp_30.shift(48)
weather_features["temp_lag_7d"] = temp_30.shift(336)

weather_features["temp_mean_24h"] = (
    temp_30.shift(2)
    .rolling(48)
    .mean()
)

weather_features["temp_mean_7d"] = (
    temp_30.shift(2)
    .rolling(336)
    .mean()
)

### A.2 — Fusion des données météo et énergie

In [13]:
import joblib
import pandas as pd
import numpy as np

X = joblib.load(
    "data_processed/X_block1_final.joblib"
)

y = joblib.load(
    "data_processed/y_block1_final.joblib"
)

print("X :", X.shape)
print("y :", y.shape)
print("Fréquence :", pd.infer_freq(X.index[:1000]))
print("Timezone X :", X.index.tz)

X : (236254, 416)
y : (236254,)
Fréquence : 30min
Timezone X : UTC


In [14]:
weather_utc = weather.copy()

idx = pd.DatetimeIndex(weather_utc.index)

idx = (
    idx
    .tz_localize(
        "Europe/Paris",
        ambiguous="NaT",
        nonexistent="NaT"
    )
    .tz_convert("UTC")
)

weather_utc.index = idx

weather_utc = weather_utc[
    ~weather_utc.index.isna()
]

if X.index.tz is None:
    weather_utc.index = (
        weather_utc.index.tz_localize(None)
    )

In [15]:
print("Énergie :", X.index.min(), "->", X.index.max())
print("Météo   :", weather_utc.index.min(), "->", weather_utc.index.max())

Énergie : 2013-01-07 23:00:00+00:00 -> 2026-06-30 21:30:00+00:00
Météo   : 2011-12-31 23:00:00+00:00 -> 2025-12-31 22:00:00+00:00


In [16]:
# Température horaire -> 30 minutes
temp_30 = (
    weather_utc["temp_france"]
    .resample("30min")
    .ffill()
)

weather_features = pd.DataFrame(index=temp_30.index)

weather_features["temp_lag_1h"] = temp_30.shift(2)
weather_features["temp_lag_24h"] = temp_30.shift(48)
weather_features["temp_lag_7d"] = temp_30.shift(336)

weather_features["temp_mean_24h"] = (
    temp_30.shift(2)
    .rolling(48)
    .mean()
)

weather_features["temp_mean_7d"] = (
    temp_30.shift(2)
    .rolling(336)
    .mean()
)

T = weather_features["temp_lag_1h"]

weather_features["heating_degree"] = (
    15 - T
).clip(lower=0)

weather_features["cooling_degree"] = (
    T - 22
).clip(lower=0)

In [17]:
X_weather = (
    X
    .join(weather_features, how="inner")
    .dropna()
)

y_weather = y.loc[X_weather.index]

print("X initial :", X.shape)
print("X météo   :", X_weather.shape)

print(
    X_weather[
        weather_features.columns
    ].head()
)

X initial : (236254, 416)
X météo   : (227567, 423)
                           temp_lag_1h  temp_lag_24h  temp_lag_7d  \
Date et Heure                                                       
2013-01-07 23:00:00+00:00         5.94          7.12         8.50   
2013-01-07 23:30:00+00:00         5.94          7.12         8.50   
2013-01-08 00:00:00+00:00         5.64          7.12         8.40   
2013-01-08 00:30:00+00:00         5.64          7.12         8.40   
2013-01-08 01:00:00+00:00         5.34          7.06         8.44   

                           temp_mean_24h  temp_mean_7d  heating_degree  \
Date et Heure                                                            
2013-01-07 23:00:00+00:00       7.066667      7.540893            9.06   
2013-01-07 23:30:00+00:00       7.039167      7.532381            9.06   
2013-01-08 00:00:00+00:00       7.008333      7.523869            9.36   
2013-01-08 00:30:00+00:00       6.977500      7.515357            9.36   
2013-01-08 01:00:00+

In [18]:
joblib.dump(
    X_weather,
    "data_processed/X_block3_weather.joblib"
)

joblib.dump(
    y_weather,
    "data_processed/y_block3_weather.joblib"
)

['data_processed/y_block3_weather.joblib']

In [19]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error
import time
import os

fit_mask = X_weather.index < "2023-01-01"

dev_mask = (
    (X_weather.index >= "2023-01-01")
    & (X_weather.index < "2024-01-01")
)

datasets = {
    "Without weather": X_weather[X.columns],
    "With weather": X_weather
}

results = []

for name, X_tmp in datasets.items():

    path = (
        "models/block3_"
        + name.lower().replace(" ", "_")
        + ".joblib"
    )

    if os.path.exists(path):
        model = joblib.load(path)

    else:
        start = time.time()

        model = Pipeline([
            ("scale", StandardScaler()),
            ("ridge", Ridge(
                alpha=0.4,
                solver="lsqr",
                tol=1e-4
            ))
        ])

        model.fit(
            X_tmp.loc[fit_mask],
            y_weather.loc[fit_mask]
        )

        joblib.dump(model, path)

        print(
            name,
            f"{time.time() - start:.1f}s"
        )

    pred = model.predict(
        X_tmp.loc[dev_mask]
    )

    y_dev = y_weather.loc[dev_mask]

    results.append([
        name,
        np.sqrt(
            mean_squared_error(y_dev, pred)
        ),
        mean_absolute_error(y_dev, pred)
    ])

weather_results = pd.DataFrame(
    results,
    columns=["Model", "RMSE", "MAE"]
)

print(weather_results)

             Model        RMSE         MAE
0  Without weather  242.759290  177.781088
1     With weather  242.038569  177.373509


### Partie C — Amélioration guidée par la théorie

La température améliore légèrement les performances.

Cependant, son effet sur la consommation n'est pas purement linéaire : il dépend notamment de l'heure et de la saison.

On teste donc quelques transformations non linéaires et interactions température-calendrier.

In [20]:
X_weather_plus = X_weather.copy()

T = X_weather_plus["temp_lag_1h"]

# Non-linéarité
X_weather_plus["temp_squared"] = T ** 2
X_weather_plus["temp_cubic"] = T ** 3

# Interaction avec l'heure
X_weather_plus["temp_hour_sin"] = (
    T * X_weather_plus["hour_sin"]
)

X_weather_plus["temp_hour_cos"] = (
    T * X_weather_plus["hour_cos"]
)

# Interaction avec la saison
X_weather_plus["temp_month_sin"] = (
    T * X_weather_plus["month_sin"]
)

X_weather_plus["temp_month_cos"] = (
    T * X_weather_plus["month_cos"]
)

In [21]:
path = "models/block3_weather_nonlinear.joblib"

if os.path.exists(path):
    ridge_weather_plus = joblib.load(path)

else:
    start = time.time()

    ridge_weather_plus = Pipeline([
        ("scale", StandardScaler()),
        ("ridge", Ridge(
            alpha=0.4,
            solver="lsqr",
            tol=1e-4
        ))
    ])

    ridge_weather_plus.fit(
        X_weather_plus.loc[fit_mask],
        y_weather.loc[fit_mask]
    )

    joblib.dump(
        ridge_weather_plus,
        path
    )

    print(
        f"Training: {time.time()-start:.1f}s"
    )

In [22]:
pred_plus = ridge_weather_plus.predict(
    X_weather_plus.loc[dev_mask]
)

rmse_plus = np.sqrt(
    mean_squared_error(
        y_weather.loc[dev_mask],
        pred_plus
    )
)

mae_plus = mean_absolute_error(
    y_weather.loc[dev_mask],
    pred_plus
)

comparison = pd.DataFrame({
    "Model": [
        "Without weather",
        "Weather",
        "Weather + nonlinear"
    ],
    "RMSE": [
        242.759290,
        242.038569,
        rmse_plus
    ],
    "MAE": [
        177.781088,
        177.373509,
        mae_plus
    ]
})

print(comparison)

                 Model        RMSE         MAE
0      Without weather  242.759290  177.781088
1              Weather  242.038569  177.373509
2  Weather + nonlinear  241.850797  177.272169


### Conclusion de l'Étape 1

L'ajout de variables météorologiques améliore légèrement les performances du modèle.

Le RMSE passe de :

$$
242.76
$$

sans météo à :

$$
242.04
$$

avec les variables météorologiques de base.

L'ajout de transformations non linéaires et d'interactions améliore encore légèrement le résultat :

$$
RMSE = 241.85
$$

Le gain total reste modeste, d'environ :

$$
0.37\%
$$

sur le RMSE.

Les variables météorologiques sont donc conservées pour la suite du bloc, mais aucun feature engineering météo supplémentaire n'est ajouté afin d'éviter une complexité inutile.

## Étape 2 — Split Conformal Prediction
### Intervalles de prédiction conformes

### Partie A — Mise en pratique

On sépare les données temporellement :

$$
Train < 2024
$$

$$
Calibration = 2024
$$

$$
Validation = 2025
$$

Le modèle est entraîné uniquement sur le passé. Les erreurs de 2024 servent ensuite à calibrer la largeur de l'intervalle.

In [23]:
joblib.dump(
    X_weather_plus,
    "data_processed/X_block3_weather_nonlinear.joblib"
)

['data_processed/X_block3_weather_nonlinear.joblib']

In [24]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
import os
import time
import joblib
import numpy as np

train = X_weather_plus.index < "2024-01-01"

cal = (
    (X_weather_plus.index >= "2024-01-01")
    & (X_weather_plus.index < "2025-01-01")
)

val = (
    (X_weather_plus.index >= "2025-01-01")
    & (X_weather_plus.index < "2026-01-01")
)

path = "models/block3_conformal_ridge.joblib"

if os.path.exists(path):

    conformal_model = joblib.load(path)

else:

    start = time.time()

    conformal_model = Pipeline([
        ("scale", StandardScaler()),
        ("ridge", Ridge(
            alpha=0.39949,
            solver="lsqr",
            tol=1e-4
        ))
    ])

    conformal_model.fit(
        X_weather_plus.loc[train],
        y_weather.loc[train]
    )

    joblib.dump(conformal_model, path)

    print(
        f"Training : {time.time()-start:.1f}s"
    )

In [25]:
pred_cal = conformal_model.predict(
    X_weather_plus.loc[cal]
)

scores_cal = np.abs(
    y_weather.loc[cal].values
    - pred_cal
)

In [26]:
alpha = 0.10

n = len(scores_cal)

level = np.ceil(
    (n + 1) * (1 - alpha)
) / n

level = min(level, 1.0)

q_conformal = np.quantile(
    scores_cal,
    level,
    method="higher"
)

print("N calibration :", n)
print("Quantile conforme :", q_conformal)

N calibration : 17568
Quantile conforme : 314.1135121637344


In [27]:
pred_val = conformal_model.predict(
    X_weather_plus.loc[val]
)

lower = pred_val - q_conformal
upper = pred_val + q_conformal

y_val = y_weather.loc[val].values

coverage = np.mean(
    (y_val >= lower)
    & (y_val <= upper)
)

avg_width = np.mean(
    upper - lower
)

rmse = np.sqrt(
    np.mean((y_val - pred_val) ** 2)
)

print("RMSE 2025 :", rmse)
print("Coverage :", coverage)
print("Average width :", avg_width)

RMSE 2025 : 209.66140891106903
Coverage : 0.8945595707027459
Average width : 628.2270243274688


In [28]:
conformal_results = {
    "target_coverage": 0.90,
    "q_conformal": q_conformal,
    "coverage_2025": coverage,
    "average_width": avg_width,
    "rmse_2025": rmse
}

joblib.dump(
    conformal_results,
    "models/block3_split_conformal_results.joblib"
)

['models/block3_split_conformal_results.joblib']

### Partie C — Analyse

La couverture cible était :

$$
90\%
$$

La couverture observée en 2025 est :

$$
89.46\%
$$

soit un écart de seulement :

$$
-0.54
$$

point.

L'intervalle moyen est :

$$
[\hat y - 314.11,\hat y + 314.11]
$$

avec une largeur moyenne de :

$$
628.23
$$

Le résultat est proche de la couverture nominale, mais légèrement sous-calibré.

Une explication possible est la dépendance temporelle et le changement de distribution entre la période de calibration 2024 et la validation 2025.

La prochaine étape consiste donc à vérifier si cette sous-couverture est concentrée sur certaines périodes.

In [29]:
diag = pd.DataFrame({
    "y": y_val,
    "lower": lower,
    "upper": upper
}, index=X_weather_plus.loc[val].index)

diag["covered"] = (
    (diag["y"] >= diag["lower"])
    & (diag["y"] <= diag["upper"])
)

monthly_coverage = (
    diag
    .groupby(diag.index.month)["covered"]
    .mean()
)

print(monthly_coverage)

Date et Heure
1     0.861559
2     0.877976
3     0.852823
4     0.848611
5     0.899866
6     0.925000
7     0.958333
8     0.966398
9     0.911806
10    0.850806
11    0.873611
12    0.905724
Name: covered, dtype: float64


## Étape 3 — Conditional Conformal Prediction
### Intervalles adaptés à la saison

### Partie A — Mise en pratique

La dispersion des erreurs varie selon les mois.

On calcule donc un quantile conforme différent pour chaque mois à partir de la calibration 2024.

In [30]:
def conformal_q(scores, alpha=0.10):
    n = len(scores)

    level = np.ceil(
        (n + 1) * (1 - alpha)
    ) / n

    return np.quantile(
        scores,
        min(level, 1.0),
        method="higher"
    )


cal_index = X_weather_plus.loc[cal].index
val_index = X_weather_plus.loc[val].index

cal_scores = pd.DataFrame({
    "score": scores_cal,
    "month": cal_index.month
})

q_month = (
    cal_scores
    .groupby("month")["score"]
    .apply(lambda x: conformal_q(x.values))
)

print(q_month)

month
1     361.310119
2     296.851268
3     356.701585
4     400.959579
5     327.249489
6     265.646460
7     246.480405
8     225.021996
9     301.531353
10    357.223240
11    357.811467
12    328.602074
Name: score, dtype: float64


In [31]:
q_val = pd.Series(
    val_index.month,
    index=val_index
).map(q_month).values

lower_cond = pred_val - q_val
upper_cond = pred_val + q_val

covered_cond = (
    (y_val >= lower_cond)
    & (y_val <= upper_cond)
)

coverage_cond = covered_cond.mean()

width_cond = np.mean(
    upper_cond - lower_cond
)

print("Coverage conditionnelle :", coverage_cond)
print("Largeur moyenne :", width_cond)

Coverage conditionnelle : 0.8951875321116629
Largeur moyenne : 637.6435405305295


In [32]:
monthly_cond = (
    pd.DataFrame({
        "covered": covered_cond
    }, index=val_index)
    .groupby(val_index.month)
    ["covered"]
    .mean()
)

print(monthly_cond)

Date et Heure
1     0.901210
2     0.860119
3     0.885753
4     0.915278
5     0.911962
6     0.874306
7     0.895161
8     0.876344
9     0.898611
10    0.896505
11    0.907639
12    0.916498
Name: covered, dtype: float64


In [33]:
comparison_cp = pd.DataFrame({
    "Method": [
        "Split conformal",
        "Conditional conformal"
    ],
    "Coverage": [
        coverage,
        coverage_cond
    ],
    "Average_width": [
        avg_width,
        width_cond
    ]
})

print(comparison_cp)

                  Method  Coverage  Average_width
0        Split conformal  0.894560     628.227024
1  Conditional conformal  0.895188     637.643541


In [34]:
joblib.dump(
    {
        "q_month": q_month,
        "coverage": coverage_cond,
        "width": width_cond,
        "monthly_coverage": monthly_cond
    },
    "models/block3_conditional_conformal.joblib"
)

['models/block3_conditional_conformal.joblib']

### Conclusion de l'Étape 3

Le conformal conditionnel par mois améliore peu la couverture globale :

$$
89.46\% \rightarrow 89.52\%
$$

mais il réduit fortement les différences de couverture entre les mois.

Avec le split conformal standard, la couverture mensuelle variait approximativement entre :

$$
84.9\%
$$

et :

$$
96.6\%
$$

Avec le conformal conditionnel, elle varie entre :

$$
86.0\%
$$

et :

$$
91.6\%
$$

La couverture est donc beaucoup plus homogène au cours de l'année.

Cette amélioration se paie cependant par une légère augmentation de la largeur moyenne :

$$
628.23 \rightarrow 637.64
$$

soit environ :

$$
+1.5\%
$$

Le conformal conditionnel améliore donc surtout la stabilité saisonnière, mais pas l'efficacité globale.

## Étape 4 — Adaptive Conformal Prediction
### Intervalles adaptatifs dans le temps

### Partie A — Mise en pratique

In [35]:
alpha_target = 0.10
gamma = 0.005

alpha_t = alpha_target

scores_sorted = np.sort(scores_cal)

lower_aci = []
upper_aci = []

for i, pred in enumerate(pred_val):

    level = 1 - alpha_t

    q = np.quantile(
        scores_sorted,
        np.clip(level, 0.01, 0.999),
        method="higher"
    )

    low = pred - q
    high = pred + q

    lower_aci.append(low)
    upper_aci.append(high)

    covered = (
        y_val[i] >= low
        and y_val[i] <= high
    )

    error = 0 if covered else 1

    alpha_t += gamma * (
        alpha_target - error
    )

    alpha_t = np.clip(
        alpha_t,
        0.001,
        0.5
    )

In [36]:
lower_aci = np.array(lower_aci)
upper_aci = np.array(upper_aci)

coverage_aci = np.mean(
    (y_val >= lower_aci)
    & (y_val <= upper_aci)
)

width_aci = np.mean(
    upper_aci - lower_aci
)

print("ACI coverage :", coverage_aci)
print("ACI width :", width_aci)

ACI coverage : 0.899640349374893
ACI width : 640.098851607117


In [37]:
joblib.dump(
    {
        "coverage": coverage_aci,
        "width": width_aci,
        "gamma": gamma
    },
    "models/block3_aci_results.joblib"
)

['models/block3_aci_results.joblib']

### Conclusion — Adaptive Conformal Prediction

L'Adaptive Conformal Prediction obtient :

$$
Coverage = 89.96\%
$$

pour une couverture cible de :

$$
90\%
$$

avec une largeur moyenne de :

$$
640.10
$$

Par rapport au Split Conformal standard, l'intervalle est légèrement plus large mais la calibration est nettement meilleure.

ACI est donc retenu comme méthode principale de quantification de l'incertitude pour une série temporelle dont la distribution peut évoluer.

In [38]:
aci_diag = pd.DataFrame({
    "covered": (
        (y_val >= lower_aci)
        & (y_val <= upper_aci)
    )
}, index=val_index)

monthly_aci = (
    aci_diag
    .groupby(aci_diag.index.month)["covered"]
    .mean()
)

print(monthly_aci)

Date et Heure
1     0.896505
2     0.905506
3     0.883065
4     0.911111
5     0.904570
6     0.900000
7     0.911962
8     0.891129
9     0.895139
10    0.890457
11    0.905556
12    0.901684
Name: covered, dtype: float64


### Conclusion de l'ACI

L'Adaptive Conformal Prediction atteint une couverture globale de :

$$
89.96\%
$$

pour une cible de :

$$
90\%
$$

La couverture mensuelle reste également proche de la cible, entre environ :

$$
88.3\%
$$

et :

$$
91.2\%
$$

ACI est donc retenu comme méthode principale de quantification de l'incertitude.

## Étape 5 — Extreme Events & Extreme Value Theory
### Modélisation des pics et quantiles extrêmes

In [39]:
from scipy.stats import genpareto

daily_train = (
    y.loc[y.index < "2025-01-01"]
    .resample("D")
    .max()
    .dropna()
)

daily_2025 = (
    y.loc[
        (y.index >= "2025-01-01")
        & (y.index < "2026-01-01")
    ]
    .resample("D")
    .max()
    .dropna()
)

print("Nombre de jours :", len(daily_train))
print(daily_train.describe())

Nombre de jours : 4377
count     4377.000000
mean     59853.220014
std      11162.344500
min      41570.000000
25%      51588.000000
50%      56123.000000
75%      68115.000000
max      96272.000000
Name: Consommation (MW), dtype: float64


In [40]:
u = daily_train.quantile(0.95)

excess = (
    daily_train[daily_train > u]
    - u
)

print("Seuil u :", u)
print("Nombre de dépassements :", len(excess))
print(
    "Proportion :",
    len(excess) / len(daily_train)
)

Seuil u : 81005.0
Nombre de dépassements : 219
Proportion : 0.050034270047978065


In [41]:
xi, _, beta = genpareto.fit(
    excess.values,
    floc=0
)

print("Shape xi :", xi)
print("Scale beta :", beta)

Shape xi : -0.18164443271407135
Scale beta : 4297.338219814254


In [42]:
p_u = len(excess) / len(daily_train)

def extreme_quantile(p):

    if abs(xi) < 1e-8:
        return (
            u
            + beta * np.log(
                p_u / (1 - p)
            )
        )

    return (
        u
        + beta / xi
        * (
            (p_u / (1 - p)) ** xi
            - 1
        )
    )

for p in [0.99, 0.995, 0.999]:

    print(
        f"Quantile {p:.3f} :",
        extreme_quantile(p)
    )

Quantile 0.990 : 87004.24159853396
Quantile 0.995 : 89093.31426016439
Quantile 0.999 : 93040.05620410645


### Partie C — Stabilité du seuil extrême

En Peaks Over Threshold, le choix du seuil constitue un compromis :

- seuil trop bas : le modèle GPD peut être mal adapté ;
- seuil trop élevé : trop peu d'observations restent disponibles.

On vérifie donc si les paramètres et les quantiles extrêmes restent relativement stables lorsque le seuil varie.

In [44]:
threshold_levels = [0.90, 0.925, 0.95, 0.975]

evt_results = []

for level in threshold_levels:

    u_i = daily_train.quantile(level)

    excess_i = (
        daily_train[daily_train > u_i]
        - u_i
    )

    xi_i, _, beta_i = genpareto.fit(
        excess_i.values,
        floc=0
    )

    p_u_i = len(excess_i) / len(daily_train)

    def q_extreme(p):
        if abs(xi_i) < 1e-8:
            return u_i + beta_i * np.log(
                p_u_i / (1 - p)
            )

        return (
            u_i
            + beta_i / xi_i
            * (
                (p_u_i / (1 - p)) ** xi_i
                - 1
            )
        )

    evt_results.append([
        level,
        u_i,
        len(excess_i),
        xi_i,
        beta_i,
        q_extreme(0.99),
        q_extreme(0.995),
        q_extreme(0.999)
    ])

evt_stability = pd.DataFrame(
    evt_results,
    columns=[
        "threshold_q",
        "threshold_MW",
        "n_exceedances",
        "xi",
        "beta",
        "Q99",
        "Q995",
        "Q999"
    ]
)

print(evt_stability.round(2))

joblib.dump(
    evt_stability,
    "models/block3_evt_threshold_stability.joblib"
)

   threshold_q  threshold_MW  n_exceedances    xi     beta       Q99  \
0         0.90       76908.0            438 -0.27  6060.63  87337.17   
1         0.92       78578.0            329 -0.26  5564.21  87302.99   
2         0.95       81005.0            219 -0.18  4297.34  87004.24   
3         0.98       83557.8            110 -0.24  4260.52  87075.50   

       Q995      Q999  
0  89411.90  92977.56  
1  89386.35  92989.21  
2  89093.31  93040.06  
3  89248.15  93081.72  


['models/block3_evt_threshold_stability.joblib']

### Conclusion sur la stabilité du seuil

Les estimations des quantiles extrêmes restent très stables lorsque le seuil POT varie entre 90 % et 97.5 %.

En particulier :

$$
Q_{0.999}\approx93\,000\text{ MW}
$$

pour tous les seuils testés.

Le paramètre de forme reste également négatif :

$$
\xi<0
$$

ce qui correspond à une queue supérieure bornée dans le modèle GPD.

Le seuil de 95 % est conservé comme compromis entre niveau de seuil et nombre de dépassements disponibles.

In [45]:
q99 = extreme_quantile(0.99)
q995 = extreme_quantile(0.995)
q999 = extreme_quantile(0.999)

observed_max_2025 = daily_2025.max()

validation_evt = pd.DataFrame({
    "Quantile": [
        "Q99",
        "Q99.5",
        "Q99.9"
    ],
    "Level_MW": [
        q99,
        q995,
        q999
    ],
    "Exceedances_2025": [
        (daily_2025 > q99).sum(),
        (daily_2025 > q995).sum(),
        (daily_2025 > q999).sum()
    ],
    "Expected_approx": [
        len(daily_2025) * 0.01,
        len(daily_2025) * 0.005,
        len(daily_2025) * 0.001
    ]
})

print("Maximum journalier 2025 :", observed_max_2025)
print(validation_evt)

Maximum journalier 2025 : 87716.0
  Quantile      Level_MW  Exceedances_2025  Expected_approx
0      Q99  87004.241599                 1            3.650
1    Q99.5  89093.314260                 0            1.825
2    Q99.9  93040.056204                 0            0.365


### Conclusion — Extreme Value Theory

Le modèle POT-GPD ajusté avant 2025 estime :

$$
Q_{0.99}\approx87\,004\text{ MW}
$$

$$
Q_{0.995}\approx89\,093\text{ MW}
$$

$$
Q_{0.999}\approx93\,040\text{ MW}
$$

En 2025, le maximum journalier observé est :

$$
87\,716\text{ MW}
$$

Un seul dépassement de $Q_{0.99}$ est observé et aucun dépassement des niveaux $Q_{0.995}$ et $Q_{0.999}$.

Ces résultats sont compatibles avec une année relativement peu extrême.

Les estimations sont également très stables lorsque le seuil POT varie entre 90 % et 97.5 %, ce qui renforce la robustesse de l'estimation des quantiles extrêmes.

In [46]:
from scipy.stats import binom

print(
    "P(N_Q99 <= 1) :",
    binom.cdf(1, 365, 0.01)
)

print(
    "P(N_Q995 = 0) :",
    binom.pmf(0, 365, 0.005)
)

print(
    "P(N_Q999 = 0) :",
    binom.pmf(0, 365, 0.001)
)

P(N_Q99 <= 1) : 0.11959934854407153
P(N_Q995 = 0) : 0.16048131420416087
P(N_Q999 = 0) : 0.6940698870404745


#### Validation probabiliste des quantiles extrêmes

Sous une approximation binomiale :

$$
N_p \sim \mathcal{B}(365,1-p)
$$

on obtient :

$$
P(N_{0.99}\leq1)\approx11.96\%
$$

$$
P(N_{0.995}=0)\approx16.05\%
$$

$$
P(N_{0.999}=0)\approx69.41\%
$$

Les nombres de dépassements observés en 2025 ne sont donc pas incompatibles avec les quantiles extrêmes estimés.

Cette validation reste approximative car les maxima journaliers peuvent conserver une dépendance temporelle.

# Conclusion du Bloc 3

Le Bloc 3 a enrichi la prévision énergétique avec trois composantes supplémentaires.

La météo améliore légèrement la prédiction ponctuelle :

$$
RMSE: 242.76 \rightarrow 241.85
$$

La Conformal Prediction permet ensuite de quantifier l'incertitude.

L'Adaptive Conformal Inference obtient la meilleure calibration :

$$
Coverage = 89.96\%
$$

pour une cible de :

$$
90\%
$$

avec une couverture mensuelle beaucoup plus stable.

Enfin, la théorie des valeurs extrêmes permet d'estimer les niveaux de consommation rares :

$$
Q_{0.999}\approx93\,000\text{ MW}
$$

avec une bonne stabilité vis-à-vis du choix du seuil.

Le projet combine désormais prédiction ponctuelle, quantification de l'incertitude et analyse des événements extrêmes.